# Structured text values - JavaScript

All 12 JavaScript examples from [docs/text.md](https://platob.github.io/yggdryl/text/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and call `require`, so they need a CommonJS
JavaScript kernel such as
[IJavascript](https://github.com/n-riesco/ijavascript), with the package
installed beside the notebook:

```console
npm install yggdryl
```

In [ ]:
const assert = require('node:assert/strict')
const { json } = require('yggdryl')

const quote = json.loads('{"symbol":"AAPL","price":12.5}')

assert.equal(quote.symbol, 'AAPL')
assert.equal(quote.price, 12.5)
assert.deepEqual(json.loads(json.dumps(quote)), quote)

## What a value can be

In [ ]:
const assert = require('node:assert/strict')
const { json } = require('yggdryl')

assert.equal(json.loads('null'), null)

// A value too wide for a Number arrives as a BigInt.
assert.equal(json.loads(json.dumps(2n ** 70n)), 2n ** 70n)

// Floats keep their exact bits.
assert.ok(Object.is(json.loads(json.dumps(-0)), -0))
assert.ok(Number.isNaN(json.loads(json.dumps(NaN))))

// Bytes survive a text format.
assert.deepEqual(json.loads(json.dumps(Buffer.from([0, 1]))), Buffer.from([0, 1]))

## Reading a shape you do not control

In [ ]:
const assert = require('node:assert/strict')
const { json } = require('yggdryl')

const order = json.loads('{"symbol":"AAPL","legs":[{"price":12},{"price":13}],"venue":null}')

assert.equal(order.legs[1].price, 13)
assert.ok('venue' in order)
assert.equal(Object.entries(order).length, 3)

// Nullish coalescing treats a present null as absent, the same as get_or.
assert.equal(order.venue ?? 'XPAR', 'XPAR')
assert.equal(order.currency ?? 'EUR', 'EUR')

## Rebuilding a mapping

In [ ]:
const assert = require('node:assert/strict')
const { json } = require('yggdryl')

const order = json.loads('{"symbol":"AAPL","venue":null}')

const updated = { ...order, venue: 'XPAR', currency: 'EUR' }
assert.equal(updated.venue, 'XPAR')
assert.equal(updated.symbol, 'AAPL')

const { venue, ...trimmed } = updated
assert.ok(!('venue' in trimmed))
assert.equal(trimmed.currency, 'EUR')

// The rebuilt object still encodes.
assert.equal(json.loads(json.dumps(trimmed)).currency, 'EUR')

## A name is not a type

In [ ]:
const assert = require('node:assert/strict')
const { json, yaml } = require('yggdryl')

const tagged =
  '{"$yggdryl":{"version":1,"type":"tag","tag":"app:Trade","value":{"symbol":"AAPL"}}}'
assert.deepEqual(json.loads(tagged), {
  $yggdryl: {
    version: 1,
    type: 'tag',
    tag: 'app:Trade',
    value: { symbol: 'AAPL' },
  },
})

// A YAML application tag annotates a node; the node is what arrives.
assert.deepEqual(yaml.loads('!app:Trade {symbol: AAPL}\n'), { symbol: 'AAPL' })

## Four formats, one surface

In [ ]:
const assert = require('node:assert/strict')
const { json, toml, yaml } = require('yggdryl')

const quote = { symbol: 'AAPL' }

assert.equal(json.dumps(quote).toString(), '{"symbol":"AAPL"}')
assert.deepEqual(toml.loads(toml.dumps(quote)), quote)
assert.deepEqual(yaml.loads(yaml.dumps(quote)), quote)

// JSON and YAML hold many documents; TOML holds exactly one.
assert.equal(json.dumpAll([{ id: 1 }, { id: 2 }]).toString(), '{"id":1}\n{"id":2}\n')
assert.deepEqual(json.loadsAll('{"id":1}\n{"id":2}\n'), [{ id: 1 }, { id: 2 }])
assert.deepEqual(yaml.loadsAll('a: 1\n---\na: 2\n'), [{ a: 1 }, { a: 2 }])
assert.equal(toml.dumpAll, undefined)

## Inferring the format

In [ ]:
const assert = require('node:assert/strict')
const { codec } = require('yggdryl')

// codec.from infers the grammar from the content itself.
assert.deepEqual(codec.from('{"symbol":"AAPL"}'), { symbol: 'AAPL' })
assert.deepEqual(codec.from('symbol = "AAPL"\n'), { symbol: 'AAPL' })
assert.deepEqual(codec.from('symbol: AAPL\n'), { symbol: 'AAPL' })

// An explicit format overrides content.
assert.deepEqual(codec.from('symbol = "AAPL"\n', { format: 'toml' }), { symbol: 'AAPL' })

## Bounds on untrusted input

In [ ]:
const assert = require('node:assert/strict')
const { json } = require('yggdryl')

// maxDepth tightens the same bound the core applies.
assert.deepEqual(json.loads('{"a":[1]}', { maxDepth: 2 }), { a: [1] })

assert.throws(
  () => json.loads('{"a":[[1]]}', { maxDepth: 2 }),
  /nesting depth limit exceeded/,
)

## Failures carry a byte position

In [ ]:
const assert = require('node:assert/strict')
const { json, toml } = require('yggdryl')

assert.throws(() => json.loads('{"symbol": '), /invalid json data at byte 11/)
assert.throws(() => toml.loads('symbol = '), /invalid toml data at byte 9/)

## Through a storage handle

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const { codec, json } = require('yggdryl')

const directory = fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-text-'))
const quote = path.join(directory, 'quote.json')

json.dump({ symbol: 'AAPL' }, quote)
assert.equal(fs.readFileSync(quote).toString(), '{"symbol":"AAPL"}')
assert.deepEqual(json.load(quote), { symbol: 'AAPL' })

// codec.from takes the format from the suffix.
assert.deepEqual(codec.from(quote), { symbol: 'AAPL' })
fs.rmSync(directory, { recursive: true, force: true })

## Jinja-style placeholders

In [ ]:
const assert = require('node:assert/strict')
const { yaml } = require('yggdryl')

const variables = { ROOT: '/var/log', PORT: 8080 }
const document =
  'path: "{{ ROOT }}/app"\nport: "{{ PORT }}"\ntls: "{{ TLS | default(false) }}"\n'
const value = yaml.loads(document, { placeholders: variables })

assert.equal(value.path, '/var/log/app')
assert.equal(value.port, 8080)
assert.equal(value.tls, false)

assert.throws(() => yaml.loads('a: "{{ MISSING }}"\n', { placeholders: {} }), /MISSING/)

### The environment is a second switch

In [ ]:
const assert = require('node:assert/strict')
const { yaml } = require('yggdryl')

const document = 'h: "{{ HOME_DIR }}"\n'

// Resolving from a mapping alone: no environment access whatsoever.
assert.equal(yaml.loads(document, { placeholders: { HOME_DIR: '/supplied' } }).h, '/supplied')

// The environment, turned on explicitly, and still losing to the mapping.
const value = yaml.loads(document, {
  placeholders: { HOME_DIR: '/supplied' },
  environment: true,
})
assert.equal(value.h, '/supplied')